In [ ]:
"""R1 — `setup_biep_registry_header()` collapses the 14-line header."""
_ctx = setup_biep_registry_header()
mo.md(
    f"""
    # 📚 Leaving Cert Subject Panel — 7-tab grouped marimo (BIEP v2)

    Browse the 6 priority LC subjects side-by-side, plus an
    EN/GA comparison view. Replaces the 7 per-subject
    `notebooks/leaving_cert/<subject>.py` + `06_en_vs_ga_comparison.py`
    files (~2,657 LOC).

    **Registry**: `{_ctx["registry_summary"]}` ({_ctx["dlt_source_count"]} DLT + {_ctx["coco_app_count"]} CocoIndex + {_ctx["baml_class_count"]} BAML)
    **Default LLM**: `{_ctx["default_llm"]}` ({_ctx["enabled_models"]} enabled)
    """
)

In [ ]:
"""The filter UI — subject + level + language multiselects."""
subject_filter = mo.ui.multiselect(
    options=["mathematics", "chemistry", "geography", "english", "gaeilge", "computer_science"],
    value=["mathematics", "english"],
    label="LC subject",
)
level_filter = mo.ui.multiselect(
    options=["hl", "ol", "fl"],
    value=["hl", "ol"],
    label="Level",
)
language_filter = mo.ui.multiselect(
    options=["en", "ga"],
    value=["en", "ga"],
    label="Language",
)
mo.vstack([subject_filter, level_filter, language_filter])

In [ ]:
"""The ibis-first connection (per the BIEP v2 spec)."""
from notebooks._shared.db import connect_md

conn = connect_md()
mo.md("✓ ibis-first wired — per-subject LanceDB tables: `cianhoghlaim.lc.<subject>.<level>_<lang>`")

In [ ]:
"""The 8-tab grouped view (P1 — `mo.ui.tabs`).

Each per-subject tab queries the per-subject LC LanceDB table via ibis.
The 8th tab is the LLM "Ask the Syllabus" chat (P3).
"""
subject_tabs = mo.ui.tabs(
    {
        "1. Mathematics": _lc_query(
            conn, "mathematics", subject_filter, level_filter, language_filter, mo
        ),
        "2. Chemistry": _lc_query(
            conn, "chemistry", subject_filter, level_filter, language_filter, mo
        ),
        "3. Geography": _lc_query(
            conn, "geography", subject_filter, level_filter, language_filter, mo
        ),
        "4. Gaeilge (EN/GA)": _lc_query(
            conn, "gaeilge", subject_filter, level_filter, language_filter, mo
        ),
        "5. English (EN/GA)": _lc_query(
            conn, "english", subject_filter, level_filter, language_filter, mo
        ),
        "6. Computer Science": _lc_query(
            conn, "computer_science", subject_filter, level_filter, language_filter, mo
        ),
        "7. EN/GA Comparison": _en_vs_ga_query(conn, subject_filter, level_filter, mo),
        "8. 🤖 Ask BAML": _llm_tab(mo),  # P3 — LLM-assisted analysis tab
    }
)
subject_tabs

In [ ]:
"""Canonical ibis query: per-subject topic counts for the given filters.

Returns a pandas DataFrame for marimo's table rendering.
"""

In [ ]:
"""EN/GA comparison query — the old `06_en_vs_ga_comparison.py` content."""

In [ ]:
"""P3 — LLM-assisted analysis tab via mo.ui.chat + mo.ai.llm.openai()."""
_chat = llm_chat_with_prompts(
    system_message=(
        "You are the BIEP v2 Leaving Cert Subject Panel assistant. "
        "You have access to the 6 priority LC subjects (Mathematics, "
        "Chemistry, Geography, English, Gaeilge, Computer Science) across "
        "Higher/Ordinary/Foundation levels and EN/GA languages. When the "
        "user asks about a specific topic, refer to the cianfhoghlaim.lc."
        "<subject>_topics tables."
    ),
    prompts=[
        "📚 Summarise the Mathematics Higher EN learning outcomes for 'algebra'",
        "🔍 Find the Irish-language equivalent for the Chemistry topic 'atomic structure'",
        "📊 Compare EN vs GA topic frequency for English Higher",
        "🎯 What are the 5 most-tested topics on the Gaeilge Higher exam?",
        "🌐 Translate the Computer Science HL topic 'algorithms' into Irish",
    ],
)
mo.vstack([mo.md("## 🤖 Ask the LC Subject Panel (via litellm)"), _chat])